<a href="https://colab.research.google.com/github/VictorJorge5/panel-vuelos/blob/main/Lambda_Ingest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import boto3
import joblib
import os
import pandas as pd
from datetime import datetime, timezone

s3 = boto3.client('s3')

def extraer_clima(iata, ts, meteo):
    d = [0.0, 0.0, 0.0, 10000.0, 0.0, 15.0, 0.0]
    if not iata or iata not in meteo:
        return d
    try:
        val_ts = ts or datetime.now(timezone.utc).timestamp()
        clave = datetime.fromtimestamp(val_ts, timezone.utc).strftime('%Y-%m-%dT%H:00')
        return meteo[iata].get(clave, d)
    except:
        return d

def lambda_handler(event, context):
    print("--- INICIO DE PROCESAMIENTO DETECTIVE ---")
    try:
        bucket = event['Records'][0]['s3']['bucket']['name']
        key = event['Records'][0]['s3']['object']['key']

        # Criba de seguridad interna: Si el archivo no viene de raw/, ignoramos
        if not key.startswith("raw/"):
            print(f"⏩ Archivo fuera de raw/ ({key}). Ignorando para evitar bucles.")
            return {"status": "ignored", "reason": "not_in_raw"}

        path = '/tmp/model.joblib'
        if not os.path.exists(path):
            s3.download_file(bucket, 'model/modelo_vuelos_final.joblib', path)
        model_data = joblib.load(path)

        raw = json.loads(s3.get_object(Bucket=bucket, Key=key)['Body'].read().decode('utf-8'))
        meteo = raw.get('meteo_detallada', {})
        preds = {}

        fuentes = [
            ('aire', raw.get('vuelos_en_aire', [])),
            ('llegadas', raw.get('llegadas_programadas', [])),
            ('salidas', raw.get('salidas_programadas', []))
        ]

        for tipo, lista in fuentes:
            for item in lista:
                try:
                    if tipo == 'aire':
                        f_id = item.get('callsign')
                        orig, dest = item.get('origen'), item.get('destino')
                        linea = item.get('aerolinea_iata', 'N/A')
                        ts = item.get('time_details')
                    else:
                        f_obj = item.get('flight', {})
                        ident = f_obj.get('identification', {}) or {}
                        f_id = ident.get('callsign') or ident.get('number', {}).get('default') or item.get('flight_number')

                        orig = f_obj.get('airport', {}).get('origin', {}).get('code', {}).get('iata')
                        dest = item.get('target_apt')
                        linea = f_obj.get('airline', {}).get('code', {}).get('iata')

                        f_time = f_obj.get('time', {}) or {}
                        f_sch = f_time.get('scheduled', {}) or {}
                        ts = f_sch.get('arrival' if tipo=='llegadas' else 'departure')

                    if not f_id: continue
                    f_id = str(f_id).strip().upper()

                    c_o = extraer_clima(orig, ts, meteo)
                    c_d = extraer_clima(dest, ts, meteo)

                    e_o = model_data['le_orig'].transform([orig])[0] if orig in model_data['le_orig'].classes_ else 0
                    e_d = model_data['le_dest'].transform([dest])[0] if dest in model_data['le_dest'].classes_ else 0
                    e_l = model_data['le_carrier'].transform([linea])[0] if linea in model_data['le_carrier'].classes_ else 0

                    df = pd.DataFrame([[
                        float(c_o[0]), float(c_o[1]), float(c_o[3]), float(c_o[4]), float(c_o[5]),
                        float(c_d[0]), float(c_d[1]), float(c_d[3]), float(c_d[4]), float(c_d[5]),
                        int(e_o), int(e_d), int(e_l)
                    ]], columns=model_data['features'])

                    prob = float(model_data['modelo'].predict_proba(df)[0][1])
                    n, c, i = ("BAJA", "green", " 🟢  Baja") if prob < 0.10 else (("MEDIA", "orange", " 🟠  Media") if prob < 0.20 else ("ALTA", "red", " 🔴  Alta"))

                    preds[f_id] = {"prob_texto": f"{prob:.1%}", "alerta": n, "color": c, "icono": i}

                except: continue

        raw['predicciones_ia'] = preds
        if 'metadata' not in raw: raw['metadata'] = {}
        raw['metadata']['procesado_ia_utc'] = datetime.now(timezone.utc).isoformat()

        # 1. Guardamos el archivo estático fijo que consume tu aplicación de Streamlit
        s3.put_object(
            Bucket=bucket,
            Key="predictions/latest_results.json",
            Body=json.dumps(raw, indent=4),
            ContentType='application/json'
        )
        print(" Archivo fijo actualizado en predictions/latest_results.json")

        # 2. Guardamos la copia de auditoría ordenada dentro de predictions/archive/
        timestamp_ahora = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
        ruta_historico_predicciones = f"predictions/archive/results_{timestamp_ahora}.json"

        s3.put_object(
            Bucket=bucket,
            Key=ruta_historico_predicciones,
            Body=json.dumps(raw, indent=4),
            ContentType='application/json'
        )
        print(f" Histórico de auditoría guardado en: {ruta_historico_predicciones}")

        return {"status": "success"}

    except Exception as e:
        print(f"❌ ERROR: {str(e)}")
        return {"status": "error"}